# 06. Phase 4 인용 오류 수정·캐시 재검증

이 노트북을 **기존 Phase 4 폴더의 `notebooks/`에 저장**하고 기존 가상환경 커널로 실행하세요.
별도의 ZIP을 풀거나 파일을 수동으로 덮어쓸 필요가 없습니다.

확인된 실패는 “인용문이 청크 원문과 정확히 일치하지 않습니다”입니다.
원래 결과 JSON에는 모델 응답 본문이 없어 어떤 문자가 달랐는지는 아직 확정하지 못했습니다.
여기서는 로컬 캐시의 같은 응답을 꺼내 원래 인용문을 검사합니다. **API 키·추가 모델 호출·추가 API 사용료가 필요하지 않습니다.**

수정은 공백·줄바꿈만 다른 인용이 **같은 청크 안에서 한 위치로 복원될 때** 원문 부분문자열로 되돌린 뒤 기존 엄격 검사를 다시 실행합니다.
숫자·단어·단위·부가세·문장부호는 바꾸지 않습니다. 다른 문서나 다른 청크를 찾아 인용을 대체하지 않습니다.
복원이 모호하거나 실제 내용이 다르면 답변은 계속 차단하고 세부 진단을 표시합니다.

적용 전 파일 해시를 확인하고 이전 코드를 자동 백업합니다. 기존 원문·인덱스·모델 캐시·실패 결과는 보존합니다.
수정 대상은 답변·요약의 인용 연결 코드와 검사 테스트입니다. 기존 05 노트북은 설정과 출력 기록을 유지하면서 오류 표시 부분만 보완합니다.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = None  # 다른 위치라면 기존 rfp_phase4_local_20261111 폴더 Path 지정
FAILED_RESULT = 'phase4_outputs/runs/20261111T084431_338782Z_d966ed59/result_003.json'

if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd().parent,
                  Path.cwd() / 'rfp_phase4_local_20261111']
    ROOT = next((p.resolve() for p in candidates
                 if (p / 'rfp_phase4/runtime.py').is_file()
                 and (p / 'index/index_meta.json').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None:
    raise FileNotFoundError('이 파일을 기존 Phase 4 폴더의 notebooks/에 넣거나 PROJECT_ROOT를 지정하세요.')
print('Python:', sys.executable)
print('기존 프로젝트:', ROOT)
print('재검증할 결과:', ROOT / FAILED_RESULT)
print('외부 API 호출: 0회')


Python: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/bin/python
기존 프로젝트: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111
재검증할 결과: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/runs/20261111T084431_338782Z_d966ed59/result_003.json
외부 API 호출: 0회


## 1. 수정 코드 적용

다음 셀에는 수정 파일이 포함되어 있습니다. 이미 적용한 파일은 건너뛰며 직접 수정한 코드와 충돌하면 덮어쓰지 않고 중단합니다.
`modified_files`와 `backup_directory`를 확인하세요. 데이터·인덱스를 재구축하지 않습니다.


In [2]:
INSTALLER_SOURCE = '"""Apply the reviewed local Phase4 citation fix; stdlib only, no API calls."""\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport sys\n\n\nPATCH_ID = \'phase4-citation-fix-v1\'\nALLOWED_PATHS = {\'rfp_phase4/citation_recovery.py\',\'rfp_phase4/rag.py\',\n                 \'rfp_phase4/summarization.py\',\'tests/test_phase4_citation_recovery.py\'}\n\n\ndef _sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef apply_patch(project_root, patch_root=None, *, embedded_files=None, manifest=None):\n    project = Path(project_root).expanduser().resolve()\n    patch_root = Path(patch_root or Path(__file__).resolve().parent)\n    if manifest is None:\n        manifest = json.loads((patch_root/\'patch_manifest.json\').read_text(encoding=\'utf-8\'))\n    if manifest.get(\'patch_id\') != PATCH_ID or set(manifest.get(\'files\',{})) != ALLOWED_PATHS:\n        raise ValueError(\'패치 파일 목록 또는 버전이 올바르지 않습니다.\')\n    if not (project/\'rfp_phase4/runtime.py\').is_file() or not (project/\'index/index_meta.json\').is_file():\n        raise ValueError(\'기존 rfp_phase4_local_20261111 최상위 폴더를 지정하세요.\')\n    staged, skipped = {}, []\n    # Validate every target before writing anything. User edits are never overwritten silently.\n    for relative, item in manifest[\'files\'].items():\n        target = (project/relative).resolve()\n        target.relative_to(project)\n        content = embedded_files[relative].encode(\'utf-8\') if embedded_files is not None else (patch_root/\'payload\'/relative).read_bytes()\n        if _sha(content) != item[\'new_sha256\']:\n            raise ValueError(\'패치 내용의 해시가 다릅니다: \'+relative)\n        current = target.read_bytes() if target.is_file() else None\n        actual = _sha(current) if current is not None else None\n        if actual == item[\'new_sha256\']:\n            skipped.append(relative); continue\n        if actual != item[\'old_sha256\']:\n            raise ValueError(\'직접 수정했거나 다른 버전인 파일이 있어 적용을 중단했습니다: \'+relative)\n        staged[relative] = content\n    notebook = project/\'notebooks/05_younggwan_phase4_grounded_answers.ipynb\'\n    notebook_modified = False\n    notebook_content = None\n    if notebook.is_file():\n        nb = json.loads(notebook.read_text(encoding=\'utf-8\'))\n        anchor = "    if RUN_OUTPUT_DIR is not None:\\n        RESULT_SEQUENCE += 1"\n        addition = ("    validation_detail = result.get(\'validation\') or {}\\n"\n            "    if validation_detail.get(\'errors\'):\\n"\n            "        print(\'출처 검증 세부 오류:\')\\n"\n            "        show_json(validation_detail[\'errors\'])\\n"\n            "        print(\'검증되지 않은 인용 데이터:\')\\n"\n            "        show_json([row for row in validation_detail.get(\'citation_diagnostics\', [])\\n"\n            "                   if row.get(\'outcome\') not in {\'already_exact\', \'whitespace_only\'}])\\n"\n            "    recovery = validation_detail.get(\'citation_recovery\') or {}\\n"\n            "    if recovery.get(\'corrected_citations\'):\\n"\n            "        print(\'공백 차이를 원문으로 복원한 인용 수:\', recovery[\'corrected_citations\'])\\n")\n        for cell in nb.get(\'cells\',[]):\n            source=cell.get(\'source\',\'\')\n            text=\'\'.join(source) if isinstance(source,list) else source\n            if cell.get(\'cell_type\')==\'code\' and \'def show_result(result):\' in text and \'validation_detail =\' not in text and anchor in text:\n                cell[\'source\']=text.replace(anchor,addition+anchor,1)\n                notebook_modified=True\n        if notebook_modified:\n            notebook_content=(json.dumps(nb,ensure_ascii=False,indent=2)+\'\\n\').encode(\'utf-8\')\n    backup = None\n    modified = list(staged)\n    if notebook_content is not None:\n        relative=str(notebook.relative_to(project));staged[relative]=notebook_content;modified.append(relative)\n    written=[]\n    try:\n        if staged:\n            stamp=datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S_%fZ\')\n            backup=project/\'phase4_outputs/patch_backups\'/stamp\n            backup.mkdir(parents=True)\n            for relative, content in staged.items():\n                target=project/relative\n                if target.is_file():\n                    saved=backup/relative;saved.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(target,saved)\n                target.parent.mkdir(parents=True,exist_ok=True)\n                temporary=target.with_name(target.name+\'.citation-patch-tmp\')\n                temporary.write_bytes(content);temporary.replace(target);written.append(relative)\n    except Exception:\n        for relative in reversed(written):\n            target=project/relative; saved=backup/relative\n            if saved.is_file():shutil.copy2(saved,target)\n            else:target.unlink(missing_ok=True)\n        raise\n    report={\'patch_id\':PATCH_ID,\'modified_files\':modified,\'already_applied_files\':skipped,\n            \'backup_directory\':str(backup) if backup else None,\'notebook_diagnostics_added\':notebook_modified,\n            \'data_or_index_modified\':False,\'cache_modified\':False,\'api_calls\':0,\n            \'restart_phase4_notebook_kernel_before_reusing_old_sessions\':True}\n    if staged:\n        report_path=backup/\'patch_report.json\'\n        report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return report\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--project\',type=Path,default=Path.cwd())\n    args=parser.parse_args()\n    print(json.dumps(apply_patch(args.project),ensure_ascii=False,indent=2))\n\n\nif __name__==\'__main__\':main()\n'
PATCH_MANIFEST = {'patch_id': 'phase4-citation-fix-v1', 'files': {'rfp_phase4/citation_recovery.py': {'old_sha256': None, 'new_sha256': '5705ba107252be370b0598fdc603a628d7f6d957b9876e03dd9ccbbd747e9795'}, 'rfp_phase4/rag.py': {'old_sha256': 'b63764c36bb606f66981163b3814975f9625917abf694497745a1153528269ab', 'new_sha256': '77f8fde3e2149fef6707d6e41e42e35f75acd3cbf5eb491cf72b5720348ceb0e'}, 'rfp_phase4/summarization.py': {'old_sha256': '9c9c7eb573d837b3886b60e6d5f5deb70bf6be8369ce1e934cbdba873bf01398', 'new_sha256': '7d3c9853bb956e872ecebe1019d311f623991e601237e402f30ba51d01c6554d'}, 'tests/test_phase4_citation_recovery.py': {'old_sha256': None, 'new_sha256': '5c10c5fa5e82410e3205b603a11f959b9ec515b3e3530f4f5090f04814810b73'}}, 'policy': 'Only a unique whitespace-only match inside the originally cited chunk is recovered, then exact original source validation runs.', 'api_calls': 0, 'original_report_model_payload_available': False}
PATCH_FILES = {'rfp_phase4/citation_recovery.py': '"""Recover formatting-only quotations, then run the original strict validator.\n\nEvery non-whitespace character, the cited chunk, document, SHA and raw offsets\nremain authoritative. This never paraphrases, changes digits/punctuation,\nsearches other documents or invokes a model. Ambiguous recovery is rejected.\n"""\nfrom __future__ import annotations\n\nimport copy\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sqlite3\n\n\nRECOVERY_POLICY = \'unique_whitespace_only_within_the_cited_chunk_v1\'\n\n\ndef _compact_with_positions(text):\n    positions = [i for i, char in enumerate(text) if not char.isspace()]\n    return \'\'.join(text[i] for i in positions), positions\n\n\ndef _recover_whitespace(source, quote):\n    needle, _ = _compact_with_positions(quote)\n    haystack, positions = _compact_with_positions(source)\n    if not needle:\n        return None, \'blank_quote\'\n    start = haystack.find(needle)\n    if start < 0:\n        return None, \'non_whitespace_characters_differ\'\n    if haystack.find(needle, start + 1) >= 0:\n        return None, \'ambiguous_whitespace_match\'\n    left, right = positions[start], positions[start + len(needle) - 1] + 1\n    restored = source[left:right]\n    assert \'\'.join(restored.split()) == \'\'.join(quote.split())\n    return {\'quote\': restored, \'quote_start\': left, \'quote_end\': right}, \'whitespace_only\'\n\n\ndef validate_with_quote_recovery(registry, payload, context_chunks, allowed_doc_ids=None):\n    """A formatting recovery layer; EvidenceRegistry itself remains strict.\n\n    Diagnostic quotations are unvalidated model data, never normal answer text.\n    Cache payloads and caller objects are not mutated. A single unresolved error\n    still prevents the whole model answer from being rendered.\n    """\n    strict = registry.validate_answer(payload, context_chunks, allowed_doc_ids=allowed_doc_ids)\n    report = {\'policy\': RECOVERY_POLICY, \'corrected_citations\': 0,\n              \'additional_api_calls\': 0, \'strict_validation_initially_passed\': strict[\'valid\']}\n    strict[\'citation_recovery\'] = report\n    strict[\'citation_diagnostics\'] = []\n    if strict[\'valid\']:\n        return strict\n    try:\n        canonical = registry.validate_chunks(context_chunks, allowed_doc_ids=allowed_doc_ids)\n    except (TypeError, ValueError, KeyError):\n        return strict\n    if not isinstance(payload, dict) or not isinstance(payload.get(\'claims\'), list):\n        return strict\n    repaired = copy.deepcopy(payload)\n    context = {chunk[\'chunk_id\']: chunk for chunk in canonical}\n    diagnostics = []\n    corrections = 0\n    for claim_index, claim in enumerate(repaired[\'claims\'], 1):\n        if not isinstance(claim, dict) or not isinstance(claim.get(\'citations\'), list):\n            continue\n        for citation_index, citation in enumerate(claim[\'citations\'], 1):\n            if not isinstance(citation, dict):\n                continue\n            cid, quote = citation.get(\'chunk_id\'), citation.get(\'quote\')\n            item = {\'claim_index\': claim_index, \'citation_index\': citation_index,\n                    \'claim_doc_id\': claim.get(\'doc_id\'), \'chunk_id\': cid,\n                    \'model_quote\': quote, \'quote_is_model_data_not_a_verified_claim\': True}\n            chunk = context.get(cid) if isinstance(cid, str) else None\n            if chunk is None:\n                item[\'outcome\'] = \'chunk_not_in_supplied_context\'\n            elif claim.get(\'doc_id\') != chunk[\'doc_id\']:\n                item[\'outcome\'] = \'claim_and_chunk_document_differ\'\n            elif not isinstance(quote, str) or not quote.strip():\n                item[\'outcome\'] = \'invalid_quote\'\n            elif quote in chunk[\'text\']:\n                item[\'outcome\'] = \'already_exact\'\n            else:\n                recovered, outcome = _recover_whitespace(chunk[\'text\'], quote)\n                item[\'outcome\'] = outcome\n                if recovered is not None:\n                    # Replacement is a literal source substring, not a normalized citation.\n                    citation[\'quote\'] = recovered[\'quote\']\n                    item[\'restored_source_quote\'] = recovered[\'quote\']\n                    item[\'source_chunk_start\'] = recovered[\'quote_start\']\n                    item[\'source_chunk_end\'] = recovered[\'quote_end\']\n                    corrections += 1\n                else:\n                    item[\'cited_chunk_preview\'] = chunk[\'text\'][:300]\n            diagnostics.append(item)\n    validated = registry.validate_answer(repaired, canonical, allowed_doc_ids=allowed_doc_ids)\n    validated[\'citation_recovery\'] = {**report, \'corrected_citations\': corrections}\n    validated[\'citation_diagnostics\'] = diagnostics\n    return validated\n\n\ndef _cached_response(root, saved):\n    """Read only the matching cache row, with checksum and original input checks."""\n    fingerprint = (saved.get(\'generation\') or {}).get(\'request_fingerprint\')\n    if not isinstance(fingerprint, str) or len(fingerprint) != 64:\n        raise ValueError(\'저장 결과에 요청 지문이 없습니다.\')\n    cache_path = Path(root) / \'phase4_outputs/generation_cache/grounded_responses.sqlite3\'\n    if not cache_path.is_file():\n        raise FileNotFoundError(\'이 프로젝트의 기존 모델 응답 캐시가 없습니다. 추가 API 호출은 하지 않았습니다.\')\n    with sqlite3.connect(cache_path.resolve().as_uri() + \'?mode=ro\', uri=True) as connection:\n        row = connection.execute(\'SELECT request_json,response_json,checksum FROM responses WHERE fingerprint=?\',\n                                 (fingerprint,)).fetchone()\n    if row is None:\n        raise ValueError(\'이 결과와 일치하는 캐시 응답이 없습니다. 추가 API 호출은 하지 않았습니다.\')\n    if hashlib.sha256(row[0].encode()).hexdigest() != fingerprint:\n        raise ValueError(\'캐시 요청 지문이 다릅니다.\')\n    if hashlib.sha256((row[0] + \'\\n\' + row[1]).encode()).hexdigest() != row[2]:\n        raise ValueError(\'캐시 응답 무결성 검사에 실패했습니다.\')\n    request, response = json.loads(row[0]), json.loads(row[1])\n    if response.get(\'request_fingerprint\') != fingerprint:\n        raise ValueError(\'캐시 응답이 다른 요청을 가리킵니다.\')\n    mode = response.get(\'execution_mode\')\n    if mode not in {\'live_openai\',\'injected_test_client\'} or request.get(\'execution_namespace\') != mode:\n        raise ValueError(\'캐시 요청과 응답의 실제/모의 실행 구분이 다릅니다.\')\n    user = json.loads(request[\'request\'][\'input\'][1][\'content\'])\n    expected_task = \'comparison\' if saved.get(\'retrieval\', {}).get(\'comparison\') else \'qa\'\n    if user.get(\'task\') != expected_task or user.get(\'question\') != saved.get(\'retrieval\', {}).get(\'standalone_query\'):\n        raise ValueError(\'캐시의 작업 또는 질문이 저장 결과와 다릅니다.\')\n    evidence = user.get(\'evidence\', [])\n    chunks = saved.get(\'source_chunks\', [])\n    if len(evidence) != len(chunks):\n        raise ValueError(\'캐시와 저장 결과의 입력 근거 개수가 다릅니다.\')\n    for original, chunk in zip(evidence, chunks):\n        for key in [\'chunk_id\',\'doc_id\',\'sha256\',\'text\']:\n            if original.get(key) != chunk.get(key):\n                raise ValueError(\'캐시와 저장 결과의 입력 근거가 다릅니다.\')\n    generation = saved.get(\'generation\') or {}\n    if response.get(\'execution_mode\') != generation.get(\'execution_mode\') or response.get(\'live_api\') != generation.get(\'live_api\'):\n        raise ValueError(\'캐시 응답의 모의/실제 실행 구분이 다릅니다.\')\n    return response\n\n\ndef replay_saved_result(root, result_path, *, registry=None, persist=True):\n    """Revalidate an existing response without API key, SDK or network calls."""\n    root, path = Path(root).resolve(), Path(result_path).resolve()\n    saved = json.loads(path.read_text(encoding=\'utf-8\'))\n    response = _cached_response(root, saved)\n    bootstrap_index = None\n    if registry is None:\n        from .runtime import bootstrap\n        app = bootstrap(root)\n        registry, bootstrap_index = app[\'registry\'], app[\'index\']\n    try:\n        checked = validate_with_quote_recovery(registry, response[\'payload\'], saved[\'source_chunks\'],\n                                               allowed_doc_ids=saved.get(\'active_doc_ids\') or None)\n        result = copy.deepcopy(saved)\n        result[\'validation\'] = checked\n        result[\'claims\'] = checked[\'claims\']\n        result[\'missing_information\'] = checked[\'missing_information\']\n        result[\'status\'] = checked[\'status\'] if checked[\'valid\'] else \'not_generated\'\n        result[\'execution_status\'] = \'completed\' if checked[\'valid\'] else \'validation_failed\'\n        result[\'answer_generated\'] = bool(checked[\'valid\'] and checked[\'claims\'])\n        result[\'markdown\'] = registry.render(checked)\n        if checked[\'valid\'] and saved.get(\'retrieval\', {}).get(\'comparison\'):\n            from .rag import _comparison_markdown\n            scopes = saved.get(\'active_doc_ids\') or []\n            cited = {claim[\'doc_id\'] for claim in checked[\'claims\']}\n            missing = [doc_id for doc_id in scopes if doc_id not in cited]\n            if missing:\n                checked[\'status\'] = \'partial\' if checked[\'claims\'] else \'insufficient_evidence\'\n                checked[\'missing_information\'].extend(f\'선택 문서 {did}: 검증된 비교 근거 미확보\' for did in missing)\n                result[\'status\'], result[\'missing_information\'] = checked[\'status\'],checked[\'missing_information\']\n            catalog = [{\'doc_id\': did, \'file_name\': registry.documents_by_id[did].get(\'file_name\',did)}\n                       for did in scopes]\n            result[\'markdown\'] = _comparison_markdown(checked, scopes, catalog)\n        result[\'replay\'] = {\'additional_api_calls\': 0, \'original_result_path\': str(path),\n                            \'scope\': \'Offline source revalidation of the original cached model response; no new semantic quality measurement.\'}\n        result[\'generation\'][\'original_usage\'] = result[\'generation\'].get(\'usage\', {})\n        result[\'generation\'][\'usage\'] = {\'api_calls\':0,\'input_tokens\':0,\'output_tokens\':0,\n                                          \'total_tokens\':0,\'cached_input_tokens\':0,\'usage_reported\':True}\n        result[\'generation\'][\'cache_hit\'] = True\n        result[\'generation\'][\'current_execution_mode\'] = \'offline_source_revalidation\'\n        result[\'generation\'][\'request_latency_ms\'] = 0.0\n        result[\'latency_ms\'] = None\n        if persist:\n            stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S_%fZ\')\n            destination = path.with_name(path.stem + \'_revalidated_\' + stamp + \'.json\')\n            with destination.open(\'x\',encoding=\'utf-8\') as handle:\n                json.dump(result,handle,ensure_ascii=False,indent=2,allow_nan=False)\n                handle.write(\'\\n\')\n            result[\'revalidated_result_path\'] = str(destination)\n        return result\n    finally:\n        if bootstrap_index is not None:\n            bootstrap_index.close()\n', 'rfp_phase4/rag.py': '"""Grounded QA, document comparison and user-only follow-up state."""\nfrom __future__ import annotations\n\nimport json\nimport re\nimport time\n\nfrom local_rag.session import SearchSession\nfrom .citation_recovery import validate_with_quote_recovery\n\n\nDEFAULT_RETRIEVAL = {\'method\': \'bm25\', \'top_k\': 8, \'per_doc_top_k\': 6,\n                     \'fetch_k\': 40, \'mmr_lambda\': .75,\n                     \'max_context_bytes\': 42000, \'max_question_chars\': 4000}\n\n\ndef _config(value):\n    config = dict(DEFAULT_RETRIEVAL)\n    supplied = dict(value or {})\n    unknown = set(supplied) - set(config)\n    if unknown:\n        raise ValueError(\'Unknown answer retrieval configuration: \' + \', \'.join(sorted(unknown)))\n    config.update(supplied)\n    for name in [\'top_k\', \'per_doc_top_k\', \'fetch_k\', \'max_context_bytes\', \'max_question_chars\']:\n        v = config[name]\n        if isinstance(v, bool) or not isinstance(v, int) or v < 1:\n            raise ValueError(name + \' must be a positive integer\')\n    if config[\'method\'] not in {\'bm25\', \'char_tfidf\', \'rrf\', \'mmr\'}:\n        raise ValueError(\'Unknown retrieval method\')\n    if isinstance(config[\'mmr_lambda\'], bool) or not isinstance(config[\'mmr_lambda\'], (int, float)) or not 0 <= config[\'mmr_lambda\'] <= 1:\n        raise ValueError(\'mmr_lambda must be between zero and one\')\n    return config\n\n\ndef _view(chunk):\n    # This is an input-size estimate, not model tokenization or pricing.\n    return {key: chunk.get(key) for key in [\'chunk_id\', \'doc_id\', \'sha256\', \'file_name\',\n            \'locator\', \'source_method\', \'quality_flags\', \'text\']}\n\n\ndef _size(chunk):\n    return len(json.dumps(_view(chunk), ensure_ascii=False, separators=(\',\', \':\')).encode(\'utf-8\'))\n\n\ndef _pack(groups, budget):\n    """Round-robin document groups before truncation; never cut source text."""\n    selected, omitted, seen = [], [], set()\n    used = 2\n    for offset in range(max((len(group) for group in groups), default=0)):\n        for group in groups:\n            if offset >= len(group):\n                continue\n            chunk = group[offset]\n            cid = chunk[\'chunk_id\']\n            if cid in seen:\n                continue\n            seen.add(cid)\n            size = _size(chunk) + 1\n            if used + size <= budget:\n                selected.append(chunk)\n                used += size\n            else:\n                omitted.append(cid)\n    return selected, omitted, used\n\n\ndef _safe(value):\n    text = str(value).replace(\'&\', \'&amp;\').replace(\'<\', \'&lt;\').replace(\'>\', \'&gt;\')\n    for char in [\'\\\\\', \'|\', \'*\', \'_\', \'`\', \'[\', \']\']:\n        text = text.replace(char, \'\\\\\' + char)\n    return text.replace(\'\\n\', \' / \')\n\n\ndef _comparison_markdown(validation, selected_doc_ids, catalog):\n    """A deterministic document-column table; every factual cell keeps citations."""\n    from .evidence import CATEGORIES, CATEGORY_LABELS\n    by_doc = {row[\'doc_id\']: row for row in catalog}\n    docs = list(selected_doc_ids)\n    claims = validation[\'claims\']\n    rows, references = [], []\n    numbered = {}\n    for claim in claims:\n        marks = []\n        for citation in claim[\'citations\']:\n            key = citation[\'chunk_id\'], citation[\'quote\']\n            if key not in numbered:\n                numbered[key] = len(references) + 1\n                references.append(citation)\n            marks.append(f\'[{numbered[key]}]\')\n        rows.append((claim[\'doc_id\'], claim[\'category\'], _safe(claim[\'text\']) + \' \' + \'\'.join(marks)))\n    lines = [\'| 항목 | \' + \' | \'.join(_safe(by_doc.get(doc, {}).get(\'file_name\', doc)) for doc in docs) + \' |\',\n             \'| --- | \' + \' | \'.join(\'---\' for _ in docs) + \' |\']\n    for category in CATEGORIES:\n        if not any(row[1] == category for row in rows):\n            continue\n        cells = [\'; \'.join(row[2] for row in rows if row[0] == doc and row[1] == category)\n                 or \'현재 근거에서 미확인\' for doc in docs]\n        lines.append(\'| \' + CATEGORY_LABELS[category] + \' | \' + \' | \'.join(cells) + \' |\')\n    if validation[\'missing_information\']:\n        lines.extend([\'\', \'추가 확인이 필요한 항목:\', \'\'])\n        lines.extend(\'- \' + _safe(item) for item in validation[\'missing_information\'])\n    if references:\n        lines.extend([\'\', \'원문 근거:\', \'\'])\n        for number, citation in enumerate(references, 1):\n            lines.append(f\'{number}. \' + _safe(citation[\'file_name\']) + \' · \' + _safe(citation[\'locator\'])\n                         + \' — “\' + _safe(citation[\'quote\']) + \'”\')\n    lines.extend([\'\', \'인용문과 원문 위치를 대조했습니다. 인용문이 주장 전체를 뒷받침하는지는 별도 검토가 필요합니다.\'])\n    return \'\\n\'.join(lines)\n\n\nclass AnswerSession:\n    """Uses only user history, never previous assistant answers or eval labels."""\n\n    def __init__(self, index, llm, registry, config=None, active_doc_ids=None):\n        self.index, self.llm, self.registry = index, llm, registry\n        self.config = _config(config)\n        self.retrieval_session = SearchSession(index, active_doc_ids=active_doc_ids)\n\n    @property\n    def active_doc_ids(self):\n        return self.retrieval_session.active_doc_ids\n\n    @property\n    def query_history(self):\n        return self.retrieval_session.query_history\n\n    def select_documents(self, ids):\n        return self.retrieval_session.select_documents(ids)\n\n    def _empty(self, status, execution_status, message, retrieval=None):\n        return {\'status\': status, \'execution_status\': execution_status,\n                \'answer_generated\': False, \'claims\': [], \'markdown\': message,\n                \'missing_information\': [], \'active_doc_ids\': self.active_doc_ids,\n                \'retrieval\': retrieval or {}, \'source_chunks\': [], \'generation\': None,\n                \'validation\': None, \'semantic_entailment_verified\': False}\n\n    def ask(self, query, *, feature=\'qa\', candidate_doc_ids=None):\n        if not isinstance(query, str):\n            raise TypeError(\'질문은 문자열이어야 합니다.\')\n        if feature not in {\'qa\', \'fact\', \'comparison\', \'followup\', \'uncertainty\'}:\n            raise ValueError(\'전체 요약은 FullDocumentSummarizer 경로를 사용하세요.\')\n        if len(query) > self.config[\'max_question_chars\']:\n            raise ValueError(\'질문 길이가 설정 상한을 넘었습니다.\')\n        started = time.perf_counter()\n        search = self.retrieval_session.ask(query, top_k=self.config[\'top_k\'],\n            method=self.config[\'method\'], candidate_doc_ids=candidate_doc_ids)\n        if search[\'status\'] != \'retrieved\':\n            return self._empty(search[\'status\'], \'not_requested\', search.get(\'reason\') or \'문서 범위와 질문을 확인하세요.\', search)\n        comparison = feature == \'comparison\' or bool(re.search(r\'비교|차이|두\\s*사업\', query))\n        scope = self.active_doc_ids\n        if comparison and len(scope) < 2:\n            return self._empty(\'needs_clarification\', \'not_requested\',\n                               \'비교할 문서를 두 개 이상 명시적으로 선택해 주세요.\', search)\n        groups = []\n        if comparison:\n            # Each document earns a separate quota BEFORE merged context packing.\n            for doc_id in scope:\n                hits = self.index.search(search[\'standalone_query\'],\n                    top_k=self.config[\'per_doc_top_k\'], allowed_doc_ids=[doc_id],\n                    method=self.config[\'method\'], fetch_k=self.config[\'fetch_k\'],\n                    mmr_lambda=self.config[\'mmr_lambda\'])\n                groups.append(hits)\n        else:\n            # SearchSession manages scope/history; use configured candidate/MMR values.\n            hits = self.index.search(search[\'standalone_query\'], top_k=self.config[\'top_k\'],\n                allowed_doc_ids=scope or None, method=self.config[\'method\'],\n                fetch_k=self.config[\'fetch_k\'], mmr_lambda=self.config[\'mmr_lambda\'])\n            groups = [hits]\n        candidate_hits = [hit for group in groups for hit in group]\n        # Raises before API dispatch if any hit or source provenance was altered.\n        self.registry.validate_chunks(candidate_hits, allowed_doc_ids=scope or None)\n        packed, omitted, input_bytes = _pack(groups, self.config[\'max_context_bytes\'])\n        context = self.registry.validate_chunks(packed, allowed_doc_ids=scope or None)\n        represented = {c[\'doc_id\'] for c in context}\n        missing_docs = [doc for doc in scope if doc not in represented]\n        retrieval = {**search, \'hits\': candidate_hits,\n            \'comparison\': comparison, \'method\': self.config[\'method\'],\n            \'context_chunk_ids\': [c[\'chunk_id\'] for c in context],\n            \'omitted_chunk_ids\': omitted, \'context_bytes_estimate\': input_bytes,\n            \'missing_selected_doc_ids\': missing_docs,\n            \'document_quotas_applied_before_packing\': comparison,\n            \'retrieval_latency_ms\': (time.perf_counter() - started) * 1000}\n        if not context:\n            return self._empty(\'insufficient_evidence\', \'not_requested\',\n                \'현재 검색·입력 예산 안에서 인용할 원문을 확보하지 못했습니다.\', retrieval)\n        enabled = getattr(getattr(self.llm, \'config\', None), \'enabled\', False)\n        if not enabled:\n            result = self._empty(\'not_generated\', \'disabled\',\n                \'OpenAI 실행이 꺼져 있습니다. 원문 근거 후보를 확인한 뒤 ENABLE_OPENAI와 해당 실행 옵션을 켜세요.\', retrieval)\n            result[\'source_chunks\'] = context\n            return result\n        try:\n            generated = self.llm.generate(\'comparison\' if comparison else \'qa\',\n                                          search[\'standalone_query\'], context)\n        except Exception as error:\n            # No upstream API body, key, or source snippet is exposed.\n            from .llm import LLMError\n            safe_detail = str(error) if isinstance(error, LLMError) else \'\'\n            result = self._empty(\'not_generated\', \'error\',\n                \'답변 생성에 실패했습니다. API 설정·사용량 상한·모델 접근 권한을 확인하세요.\', retrieval)\n            result.update({\'source_chunks\': context, \'error_type\': type(error).__name__,\n                           \'error_detail\': safe_detail,\n                           \'latency_ms\': (time.perf_counter() - started) * 1000})\n            return result\n        validation = validate_with_quote_recovery(self.registry, generated.get(\'payload\'), context,\n                                                   allowed_doc_ids=scope or None)\n        generation_report = {key: value for key, value in generated.items() if key != \'payload\'}\n        if not validation[\'valid\']:\n            result = self._empty(\'not_generated\', \'validation_failed\',\n                \'생성 결과의 출처·형식 검사에 실패해 답변을 표시하지 않습니다.\', retrieval)\n            result.update({\'validation\': validation, \'generation\': generation_report,\n                           \'source_chunks\': context, \'latency_ms\': (time.perf_counter() - started) * 1000})\n            return result\n        # Preserve missing evidence on either selected document; never imply a complete comparison.\n        cited_docs = {claim[\'doc_id\'] for claim in validation[\'claims\']}\n        missing_comparison = [doc for doc in scope if doc not in cited_docs] if comparison else []\n        if missing_docs or missing_comparison:\n            validation = dict(validation)\n            validation[\'status\'] = \'partial\' if validation[\'claims\'] else \'insufficient_evidence\'\n            validation[\'missing_information\'] = list(validation[\'missing_information\']) + [\n                f\'선택 문서 {doc}: 현재 입력에서 검증된 비교 근거 미확보\'\n                for doc in dict.fromkeys(missing_docs + missing_comparison)]\n        return {\'status\': validation[\'status\'], \'execution_status\': \'completed\',\n                \'answer_generated\': bool(validation[\'claims\']), \'claims\': validation[\'claims\'],\n                \'missing_information\': validation[\'missing_information\'],\n                \'markdown\': (_comparison_markdown(validation, scope, self.index.get_catalog())\n                             if comparison else self.registry.render(validation)), \'active_doc_ids\': scope,\n                \'source_chunks\': context, \'retrieval\': retrieval, \'generation\': generation_report,\n                \'validation\': validation, \'usage\': generated.get(\'usage\', {}),\n                \'latency_ms\': (time.perf_counter() - started) * 1000,\n                \'semantic_entailment_verified\': False}\n', 'rfp_phase4/summarization.py': '"""전체 문서의 모든 청크를 입력하는, 출처 검증 후 결정적으로 합치는 요약.\n\n입력 청크 100% 처리와 문서 의미 100% 요약은 별개다. 이 모듈은 전자를\n계수하며 후자는 주장하지 않는다. 계획은 API 호출 없이 만들 수 있다.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nfrom collections.abc import Mapping\n\nfrom .evidence import ANSWER_SCHEMA\nfrom .citation_recovery import validate_with_quote_recovery\n\n\nTASK = \'full_document_summary_map\'\nDEFAULT_QUESTION = (\n    \'이 RFP의 제공된 부분을 한국어로 항목별 정리해 주세요. 사업 개요, 예산과 부가세, \'\n    \'사업기간, 참가자격, 요구사항, 제출 방식과 기한, 평가 기준, 산출물, \'\n    \'계약상 유의사항을 살피세요. 각 사실에는 제공된 원문에서 정확히 복사한 인용문을 붙이세요. \'\n    \'이 입력은 전체 문서의 일부일 수 있으므로 여기서 찾지 못한 항목을 문서 전체에 없다고 \'\n    \'단정하지 마세요. 원문에 쓰인 금액·단위·조건·연도 차이를 그대로 보존하세요.\'\n)\nDEFAULT_CATEGORIES = (\n    \'overview\', \'budget\', \'duration\', \'eligibility\', \'requirements\', \'submission\',\n    \'evaluation\', \'deliverables\', \'risks\',\n)\nCATEGORY_LABELS = {\n    \'overview\': \'사업 개요\', \'budget\': \'예산·부가세\', \'duration\': \'사업기간\',\n    \'eligibility\': \'참가자격\', \'requirements\': \'요구사항\', \'submission\': \'제출 방식·기한\',\n    \'evaluation\': \'평가 기준\', \'deliverables\': \'산출물\', \'risks\': \'계약상 유의사항\',\n    \'other\': \'기타\',\n}\n\n\nclass SummaryError(ValueError):\n    """잘못된 문서·배치 설정 또는 확인할 수 없는 입력의 오류."""\n\n\ndef _field(value, key, default=None):\n    return value.get(key, default) if isinstance(value, Mapping) else getattr(value, key, default)\n\n\ndef _json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\n\n\nclass FullDocumentSummarizer:\n    """모든 입력을 배치별로 요약하고 출처가 검증된 주장만 순서대로 합친다.\n\n    ``plan``은 request_input_bytes만 사용하며 네트워크를 호출하지 않는다.\n    ``summarize``는 모든 요청의 예산·승인 목록을 먼저 검증한다. 기본 배치 상한은\n    30회다. 큰 문서가 이를 넘으면 입력을 자르지 않고 차단 사유와 전체 계획을 돌려준다.\n    """\n\n    def __init__(self, index, llm, registry, config=None):\n        self.index, self.llm, self.registry = index, llm, registry\n        supplied = dict(config or {})\n        allowed = {\'max_map_calls\', \'max_batch_bytes\', \'batch_max_bytes\', \'requested_categories\', \'stop_on_error\'}\n        if set(supplied) - allowed:\n            raise SummaryError(\'지원하지 않는 전체 문서 요약 설정입니다.\')\n        llm_config = getattr(llm, \'config\', {})\n        self.max_map_calls = supplied.get(\'max_map_calls\', 30)\n        request_limit = _field(llm_config, \'max_request_bytes\', 60000)\n        if \'batch_max_bytes\' in supplied and \'max_batch_bytes\' in supplied:\n            raise SummaryError(\'max_batch_bytes와 이전 별칭을 동시에 설정할 수 없습니다.\')\n        self.batch_max_bytes = supplied.get(\'max_batch_bytes\', supplied.get(\'batch_max_bytes\', min(35000, request_limit)))\n        if isinstance(self.batch_max_bytes, bool) or not isinstance(self.batch_max_bytes, int) or self.batch_max_bytes < 1:\n            raise SummaryError(\'max_batch_bytes는 양의 정수여야 합니다.\')\n        self.batch_max_bytes = min(self.batch_max_bytes, request_limit)\n        self.max_output_tokens = _field(llm_config, \'max_output_tokens\', 3000)\n        self.requested_categories = tuple(supplied.get(\'requested_categories\', DEFAULT_CATEGORIES))\n        self.stop_on_error = supplied.get(\'stop_on_error\', True)\n        for value in [self.max_map_calls, self.batch_max_bytes, self.max_output_tokens]:\n            if isinstance(value, bool) or not isinstance(value, int) or value < 1:\n                raise SummaryError(\'요약 배치·byte·출력 상한은 양의 정수여야 합니다.\')\n        if not isinstance(self.stop_on_error, bool):\n            raise SummaryError(\'stop_on_error는 bool이어야 합니다.\')\n        if (not self.requested_categories or len(set(self.requested_categories)) != len(self.requested_categories)\n                or any(value not in CATEGORY_LABELS for value in self.requested_categories)):\n            raise SummaryError(\'requested_categories는 중복 없는 지원 항목 목록이어야 합니다.\')\n        if not callable(getattr(llm, \'request_input_bytes\', None)):\n            raise SummaryError(\'LLM의 순수 request_input_bytes 계획 함수가 필요합니다.\')\n        if not callable(getattr(llm, \'preflight_requests\', None)):\n            raise SummaryError(\'모든 배치의 사전 예산 검사 기능이 필요합니다.\')\n\n    def _request(self, question, chunks):\n        return {\'task\': TASK, \'question\': question, \'evidence\': chunks, \'schema\': ANSWER_SCHEMA}\n\n    def _bytes(self, question, chunks):\n        count = self.llm.request_input_bytes(TASK, question, chunks, schema=ANSWER_SCHEMA)\n        if isinstance(count, bool) or not isinstance(count, int) or count < 1:\n            raise SummaryError(\'요청 byte 계산이 잘못되었습니다.\')\n        return count\n\n    def plan(self, doc_id, *, question=DEFAULT_QUESTION):\n        """API 없이 전체 입력 계획을 만든다. 큰 문서도 모든 청크를 계획에 남긴다."""\n        if not isinstance(question, str) or not question.strip():\n            raise SummaryError(\'요약 질문을 입력해야 합니다.\')\n        canonical = list(self.index.resolve_doc_ids([doc_id]))\n        if len(canonical) != 1:\n            raise SummaryError(\'전체 요약은 문서 한 개를 명확하게 선택해야 합니다.\')\n        canonical = canonical[0]\n        chunks = list(self.index.get_document_chunks(canonical))\n        chunks = list(self.registry.validate_chunks(chunks, allowed_doc_ids=[canonical]))\n        ids = [chunk[\'chunk_id\'] for chunk in chunks]\n        if len(ids) != len(set(ids)):\n            raise SummaryError(\'전체 문서 입력에 동일한 청크 ID가 중복됐습니다.\')\n        if any(chunk[\'doc_id\'] != canonical for chunk in chunks):\n            raise SummaryError(\'전체 문서 입력의 문서 범위가 다릅니다.\')\n\n        batches, block_reasons = [], []\n\n        def append_batch(items):\n            if items:\n                batches.append({\'batch_id\': f\'map-{len(batches) + 1:04d}\', \'chunks\': list(items),\n                                \'request_bytes\': self._bytes(question, items)})\n\n        # Exponential growth followed by binary search keeps planning a large table-rich\n        # document close to O(total serialized input) instead of repeatedly encoding every\n        # growing prefix. Each batch is a consecutive interval of the original chunk order.\n        offset = 0\n        while offset < len(chunks):\n            first_bytes = self._bytes(question, chunks[offset:offset + 1])\n            if first_bytes > self.batch_max_bytes:\n                append_batch(chunks[offset:offset + 1])\n                block_reasons.append(\'one_or_more_source_chunks_exceed_request_byte_budget\')\n                offset += 1\n                continue\n            remaining = len(chunks) - offset\n            fits = 1\n            probe = 2\n            while probe <= remaining and self._bytes(question, chunks[offset:offset + probe]) <= self.batch_max_bytes:\n                fits = probe\n                probe *= 2\n            low, high = fits + 1, min(probe, remaining)\n            while low <= high:\n                middle = (low + high) // 2\n                if self._bytes(question, chunks[offset:offset + middle]) <= self.batch_max_bytes:\n                    fits = middle\n                    low = middle + 1\n                else:\n                    high = middle - 1\n            append_batch(chunks[offset:offset + fits])\n            offset += fits\n        if not chunks:\n            block_reasons.append(\'document_has_no_indexed_chunks\')\n        if len(batches) > self.max_map_calls:\n            block_reasons.append(\'max_map_calls_exceeded\')\n        ordered = [chunk[\'chunk_id\'] for batch in batches for chunk in batch[\'chunks\']]\n        if ordered != ids:\n            raise SummaryError(\'전체 입력 계획에서 청크 순서 또는 개수가 유실됐습니다.\')\n        total_bytes = sum(batch[\'request_bytes\'] for batch in batches)\n        fingerprint = hashlib.sha256(_json({\'doc_id\': canonical, \'question\': question,\n             \'source_chunks\': [(chunk[\'chunk_id\'], chunk[\'sha256\'], hashlib.sha256(chunk[\'text\'].encode(\'utf-8\')).hexdigest()) for chunk in chunks],\n             \'batches\': [[chunk[\'chunk_id\'] for chunk in batch[\'chunks\']] for batch in batches],\n             \'requested_categories\': self.requested_categories, \'batch_max_bytes\': self.batch_max_bytes,\n             \'max_output_tokens\': self.max_output_tokens}).encode(\'utf-8\')).hexdigest()\n        return {\'doc_id\': canonical, \'question\': question, \'source_chunk_ids\': ids,\n                \'chunk_count\': len(chunks), \'map_calls\': len(batches), \'batches\': batches,\n                \'total_request_bytes\': total_bytes,\n                \'uncached_input_bytes_upper_bound\': total_bytes,\n                \'llm_max_calls\': _field(getattr(self.llm, \'config\', {}), \'max_calls\'),\n                \'llm_max_total_input_bytes\': _field(getattr(self.llm, \'config\', {}), \'max_total_input_bytes\'),\n                \'whole_session_budget_preflight_required\': True,\n                \'max_output_tokens_upper_bound\': len(batches) * self.max_output_tokens,\n                \'byte_budget_is_token_count\': False, \'estimated_price\': None,\n                \'max_map_calls\': self.max_map_calls, \'max_batch_bytes\': self.batch_max_bytes,\n                \'batch_max_bytes\': self.batch_max_bytes,\n                \'can_execute\': not block_reasons, \'block_reasons\': list(dict.fromkeys(block_reasons)),\n                \'plan_fingerprint\': fingerprint, \'api_calls\': 0,\n                \'scope\': \'Every indexed chunk appears once, in source order. This is an input plan, not semantic summary coverage.\'}\n\n    def _empty_result(self, plan):\n        return {\'status\': \'blocked\', \'execution_status\': \'not_started\', \'doc_id\': plan[\'doc_id\'],\n                \'valid\': False, \'claims\': [], \'missing_information\': [], \'errors\': [],\n                \'answer_generated\': False, \'markdown\': \'\',\n                \'source_chunk_ids\': plan[\'source_chunk_ids\'], \'processed_chunk_ids\': [],\n                \'response_received_chunk_ids\': [], \'request_attempted_chunk_ids\': [],\n                \'input_chunk_coverage\': 0.0, \'validated_chunk_coverage\': 0.0,\n                \'batch_outcomes\': [], \'map_calls\': plan[\'map_calls\'],\n                \'plan_fingerprint\': plan[\'plan_fingerprint\'], \'preflight\': None,\n                \'aggregation_method\': \'deterministic_merge_of_source_validated_map_claims\',\n                \'semantic_coverage_verified\': False, \'semantic_entailment_verified\': False,\n                \'answer_correctness_verified\': False,\n                \'usage\': {\'api_calls\': 0, \'live_api_calls\': 0, \'mock_calls\': 0,\n                          \'input_tokens\': 0, \'output_tokens\': 0, \'total_tokens\': 0,\n                          \'usage_unknown_attempts\': 0},\n                \'input_coverage_definition\': \'Fraction of indexed chunks with a completed model/cache response. Validated coverage separately requires valid source citations.\',\n                \'scope\': \'Full-document map input and citation provenance only; semantic accuracy/completeness needs review.\'}\n\n    @staticmethod\n    def _merge_claims(claims):\n        """Only exact duplicate claim text is merged; no uncited synthesis is invented."""\n        merged, by_key = [], {}\n        for claim in claims:\n            key = claim[\'doc_id\'], claim[\'category\'], claim[\'text\']\n            if key not in by_key:\n                fresh = {**claim, \'citations\': [dict(citation) for citation in claim[\'citations\']]}\n                by_key[key] = fresh\n                merged.append(fresh)\n            else:\n                target = by_key[key][\'citations\']\n                known = {(citation[\'chunk_id\'], citation[\'quote\']) for citation in target}\n                for citation in claim[\'citations\']:\n                    pair = citation[\'chunk_id\'], citation[\'quote\']\n                    if pair not in known:\n                        target.append(dict(citation)); known.add(pair)\n        return merged\n\n    def summarize(self, doc_id, question=DEFAULT_QUESTION):\n        started = time.perf_counter()\n        plan = self.plan(doc_id, question=question)\n        result = self._empty_result(plan)\n        usage_before = self.llm.usage_report() if callable(getattr(self.llm, \'usage_report\', None)) else {}\n        if not plan[\'can_execute\']:\n            result[\'execution_status\'] = \'plan_blocked\'\n            result[\'errors\'] = plan[\'block_reasons\']\n            result[\'markdown\'] = \'전체 문서 요약을 실행하지 않았습니다. 입력을 자르지 않고 계획 상한 초과를 반환했습니다.\'\n            result[\'latency_ms\'] = (time.perf_counter() - started) * 1000\n            return result\n        requests = [self._request(question, batch[\'chunks\']) for batch in plan[\'batches\']]\n        try:\n            result[\'preflight\'] = self.llm.preflight_requests(requests)\n            if not isinstance(result[\'preflight\'], dict) or result[\'preflight\'].get(\'valid\') is not True:\n                raise SummaryError(\'전체 요청의 사전 검사가 통과하지 않았습니다.\')\n        except Exception as error:\n            result[\'execution_status\'] = \'preflight_blocked\'\n            result[\'errors\'] = [\'whole_document_preflight_failed:\' + type(error).__name__]\n            result[\'markdown\'] = \'전체 문서 요약을 실행하지 않았습니다. 활성화·문서 승인·전체 요청 예산을 확인하세요.\'\n            result[\'latency_ms\'] = (time.perf_counter() - started) * 1000\n            return result\n\n        collected, processed, responded, attempted, outcomes = [], [], [], [], []\n        halted = False\n        for batch in plan[\'batches\']:\n            chunk_ids = [chunk[\'chunk_id\'] for chunk in batch[\'chunks\']]\n            outcome = {\'batch_id\': batch[\'batch_id\'], \'chunk_ids\': chunk_ids,\n                       \'request_bytes\': batch[\'request_bytes\'], \'status\': \'skipped\',\n                       \'valid\': False, \'usage\': {}, \'live_api\': False, \'cache_hit\': False,\n                       \'api_called\': False}\n            if halted:\n                outcome[\'reason\'] = \'previous_batch_failed\'\n                outcomes.append(outcome)\n                continue\n            attempted.extend(chunk_ids)\n            try:\n                response = self.llm.generate(TASK, question, batch[\'chunks\'], schema=ANSWER_SCHEMA)\n                responded.extend(chunk_ids)\n                outcome.update({\'usage\': response.get(\'usage\', {}), \'live_api\': bool(response.get(\'live_api\')),\n                                \'cache_hit\': bool(response.get(\'cache_hit\')),\n                                \'api_called\': bool(response.get(\'live_api\')) and not bool(response.get(\'cache_hit\')),\n                                \'execution_mode\': response.get(\'execution_mode\', \'mock_or_unspecified\'),\n                                \'request_fingerprint\': response.get(\'request_fingerprint\')})\n                validated = validate_with_quote_recovery(self.registry, response[\'payload\'], batch[\'chunks\'],\n                                                          allowed_doc_ids=[plan[\'doc_id\']])\n                if validated.get(\'valid\') is not True:\n                    outcome.update({\'status\': \'invalid\', \'reason\': \'source_citation_validation_failed\',\n                                    \'validation_errors\':validated[\'errors\'],\n                                    \'citation_diagnostics\':validated.get(\'citation_diagnostics\',[])})\n                    result[\'errors\'].append(batch[\'batch_id\'] + \':source_citation_validation_failed\')\n                    halted = self.stop_on_error\n                else:\n                    outcome.update({\'status\': validated[\'status\'], \'valid\': True,\n                                    \'claim_count\': len(validated[\'claims\']),\n                                    \'batch_missing_information\': validated.get(\'missing_information\', [])})\n                    processed.extend(chunk_ids)\n                    collected.extend(validated[\'claims\'])\n            except Exception as error:\n                outcome.update({\'status\': \'error\', \'reason\': \'batch_failed:\' + type(error).__name__})\n                result[\'errors\'].append(batch[\'batch_id\'] + \':batch_failed:\' + type(error).__name__)\n                halted = self.stop_on_error\n            outcomes.append(outcome)\n\n        claims = self._merge_claims(collected)\n        covered_categories = {claim[\'category\'] for claim in claims}\n        missing = [CATEGORY_LABELS[category] + \': 검증된 근거를 확보하지 못했습니다.\'\n                   for category in self.requested_categories if category not in covered_categories]\n        # A category label does not resolve a finer uncertainty such as whether VAT is\n        # included. Without a semantic reduction/review step, preserve each map\'s own\n        # uncertainty conservatively and keep its limited-input scope visible.\n        partial_batches = [outcome for outcome in outcomes if outcome.get(\'valid\') and outcome[\'status\'] == \'partial\']\n        for outcome in outcomes:\n            if not outcome.get(\'valid\'):\n                continue\n            for label in outcome.get(\'batch_missing_information\', []):\n                missing.append(f\'부분 입력 {outcome["batch_id"]}에서 미확인: {label}\')\n        if partial_batches:\n            missing.append(\'부분 입력의 미확인 사항이 다른 입력에서 해결됐는지는 자동으로 검증하지 않았습니다.\')\n        missing = list(dict.fromkeys(missing))\n        all_valid = len(processed) == len(plan[\'source_chunk_ids\'])\n        if not all_valid:\n            missing.append(\'일부 원문 청크를 검증된 요약으로 처리하지 못했습니다. 전체 문서 요약으로 확정할 수 없습니다.\')\n        if not claims:\n            status = \'insufficient_evidence\' if all_valid else \'error\'\n        else:\n            status = \'answered\' if all_valid and not missing else \'partial\'\n        result.update({\'status\': status, \'execution_status\': \'completed\' if all_valid else \'incomplete\',\n                       \'valid\': bool(claims) or all_valid, \'claims\': claims, \'missing_information\': missing,\n                       \'processed_chunk_ids\': processed, \'response_received_chunk_ids\': responded,\n                       \'request_attempted_chunk_ids\': attempted, \'batch_outcomes\': outcomes,\n                       \'partial_batch_count\': len(partial_batches),\n                       \'input_chunk_coverage\': len(responded) / plan[\'chunk_count\'],\n                       \'validated_chunk_coverage\': len(processed) / plan[\'chunk_count\'],\n                       \'answer_generated\': any(outcome[\'valid\'] for outcome in outcomes),\n                       \'elapsed_seconds\': time.perf_counter() - started})\n        if result[\'valid\']:\n            rendered = self.registry.render(result)\n            result[\'markdown\'] = rendered + \'\\n\\n입력 청크 처리율은 의미적 요약 완전성이나 사실 정확도를 보장하지 않습니다.\'\n        else:\n            result[\'markdown\'] = \'검증된 전체 문서 요약을 만들지 못했습니다. 처리 결과와 누락 청크를 확인하세요.\'\n        result[\'latency_ms\'] = (time.perf_counter() - started) * 1000\n        if callable(getattr(self.llm, \'usage_report\', None)):\n            after = self.llm.usage_report()\n            result[\'llm_session_usage\'] = after\n            for name in [\'api_calls\', \'live_api_calls\', \'mock_calls\', \'usage_unknown_attempts\']:\n                result[\'usage\'][name] = (after[name] - usage_before.get(name, 0)\n                                         if isinstance(after.get(name), (int, float)) else None)\n            for name in [\'input_tokens\', \'output_tokens\', \'total_tokens\', \'cached_input_tokens\']:\n                key = \'reported_\' + name\n                result[\'usage\'][name] = (after[key] - usage_before.get(key, 0)\n                                        if isinstance(after.get(key), (int, float)) else None)\n            result[\'usage\'][\'scope\'] = \'Counters for this summary invocation; failed attempts may have unknown token usage.\'\n        return result\n', 'tests/test_phase4_citation_recovery.py': '"""Formatting recovery must still end in exact source validation, without API."""\nimport copy\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport sqlite3\nimport tempfile\nfrom types import SimpleNamespace\nimport unittest\nfrom unittest.mock import patch\n\nfrom test_phase4_evidence import document, block, chunk, answer\nfrom rfp_phase4.evidence import EvidenceRegistry\nfrom rfp_phase4.citation_recovery import validate_with_quote_recovery, replay_saved_result\nfrom rfp_phase4.llm import LLMConfig, OpenAIGroundedLLM\n\n\nclass RecoveryTests(unittest.TestCase):\n    def fixture(self, text=\'사업예산 :  100원.\\n사업기간 : 5개월.\'):\n        b=block(text=text); d=document(source=b); c=chunk(b)\n        return d,c,EvidenceRegistry([d],[c])\n\n    def test_original_validator_remains_strict(self):\n        d,c,reg=self.fixture()\n        payload=answer(quote=\'사업예산:100원.\')\n        self.assertFalse(reg.validate_answer(payload,[c])[\'valid\'])\n\n    def test_unique_whitespace_difference_is_replaced_by_exact_original(self):\n        d,c,reg=self.fixture()\n        payload=answer(quote=\'사업예산:100원.\')\n        original=copy.deepcopy(payload)\n        r=validate_with_quote_recovery(reg,payload,[c],[\'d1\'])\n        self.assertTrue(r[\'valid\']); self.assertEqual(payload,original)\n        quote=r[\'claims\'][0][\'citations\'][0]\n        self.assertEqual(quote[\'quote\'],\'사업예산 :  100원.\')\n        span=quote[\'source_ranges\'][0]\n        self.assertEqual(d[\'blocks\'][0][\'text\'][span[\'char_start\']:span[\'char_end\']],quote[\'quote\'])\n        self.assertEqual(r[\'citation_recovery\'][\'corrected_citations\'],1)\n        self.assertEqual(r[\'citation_recovery\'][\'additional_api_calls\'],0)\n        self.assertFalse(r[\'semantic_entailment_verified\'])\n\n    def test_line_wrap_tab_and_unicode_whitespace_are_source_restored(self):\n        d,c,reg=self.fixture(\'사업예산\\n:\\t100원.\\u00a0부가세 포함\')\n        r=validate_with_quote_recovery(reg,answer(quote=\'사업예산:100원. 부가세포함\'),[c])\n        self.assertTrue(r[\'valid\'])\n        self.assertEqual(r[\'claims\'][0][\'citations\'][0][\'quote\'],c[\'text\'])\n\n    def test_digits_currency_vat_punctuation_negation_are_never_repaired(self):\n        d,c,reg=self.fixture(\'사업예산 : 100원(부가세 포함), 면제 아님.\')\n        for quote in [\'사업예산:999원(부가세포함)\',\'사업예산:100달러(부가세포함)\',\n                      \'사업예산:100원(부가세별도)\',\'사업예산:100원[부가세포함]\', \'면제임.\']:\n            with self.subTest(quote=quote):\n                r=validate_with_quote_recovery(reg,answer(quote=quote),[c])\n                self.assertFalse(r[\'valid\']); self.assertEqual(r[\'claims\'],[])\n\n    def test_ambiguous_nonexact_recovery_is_rejected(self):\n        d,c,reg=self.fixture(\'예산 : 100원 / 예산: 100원\')\n        r=validate_with_quote_recovery(reg,answer(quote=\'예산:100원\'),[c])\n        self.assertFalse(r[\'valid\'])\n        self.assertEqual(r[\'citation_diagnostics\'][0][\'outcome\'],\'ambiguous_whitespace_match\')\n\n    def test_other_chunks_and_documents_are_not_searched(self):\n        d,c,reg=self.fixture()\n        second=block(\'b2\',text=\'예산 : 999원\')\n        d[\'blocks\'].append(second)\n        c2=chunk(second,cid=\'c2\')\n        reg=EvidenceRegistry([d],[c,c2])\n        self.assertEqual(len(reg.validate_chunks([c,c2])),2)\n        r=validate_with_quote_recovery(reg,answer(quote=\'예산:999원\'),[c,c2])\n        self.assertFalse(r[\'valid\'])\n        r=validate_with_quote_recovery(reg,answer(quote=\'사업예산:100원.\',cid=\'not-provided\'),[c])\n        self.assertFalse(r[\'valid\'])\n\n    def test_scope_escape_wrong_doc_and_tampered_source_remain_rejected(self):\n        d,c,reg=self.fixture(); p=answer(quote=\'사업예산:100원.\')\n        self.assertFalse(validate_with_quote_recovery(reg,p,[c],[])[\'valid\'])\n        self.assertFalse(validate_with_quote_recovery(reg,answer(quote=p[\'claims\'][0][\'citations\'][0][\'quote\'],did=\'other\'),[c])[\'valid\'])\n        c[\'text\']=\'가짜 원문\'; c[\'text_bytes\']=len(c[\'text\'].encode())\n        self.assertFalse(validate_with_quote_recovery(reg,p,[c])[\'valid\'])\n\n    def test_one_unrepairable_claim_blocks_other_recovered_claims(self):\n        d,c,reg=self.fixture()\n        p=answer(quote=\'사업예산:100원.\')\n        p[\'claims\']+=answer(quote=\'기간99개월\')[\'claims\']\n        r=validate_with_quote_recovery(reg,p,[c])\n        self.assertFalse(r[\'valid\']);self.assertEqual(r[\'claims\'],[])\n        self.assertEqual(r[\'citation_recovery\'][\'corrected_citations\'],1)\n        self.assertEqual(len(r[\'citation_diagnostics\']),2)\n\n    def test_exact_quotes_are_left_unchanged(self):\n        d,c,reg=self.fixture()\n        r=validate_with_quote_recovery(reg,answer(quote=\'사업예산 :  100원.\'),[c])\n        self.assertTrue(r[\'valid\']);self.assertEqual(r[\'citation_recovery\'][\'corrected_citations\'],0)\n\n    def test_schema_errors_still_fail(self):\n        d,c,reg=self.fixture()\n        for p in [None,[],{},dict(answer(),unexpected=\'field\')]:\n            self.assertFalse(validate_with_quote_recovery(reg,p,[c])[\'valid\'])\n\n\nclass ReplayTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp=tempfile.TemporaryDirectory();self.addCleanup(self.tmp.cleanup)\n        self.root=Path(self.tmp.name)\n        b=block(text=\'사업예산 :  100원.\')\n        self.d,self.c=document(source=b),chunk(b)\n        self.reg=EvidenceRegistry([self.d],[self.c])\n        self.payload=answer(quote=\'사업예산:100원.\')\n        class Responses:\n            def __init__(inner):inner.calls=0\n            def create(inner,**kwargs):\n                inner.calls+=1\n                return SimpleNamespace(status=\'completed\',output=[],output_text=json.dumps(self.payload),\n                    usage=SimpleNamespace(input_tokens=10,output_tokens=20,total_tokens=30))\n        self.responses=Responses()\n        with patch.dict(os.environ,{\'OPENAI_API_KEY\':\'offline-test-only\'}):\n            client=OpenAIGroundedLLM(self.root/\'phase4_outputs/generation_cache\',\n                config=LLMConfig(enabled=True),allowed_sources={\'d1\':\'a\'*64},client=SimpleNamespace(responses=self.responses))\n            generated=client.generate(\'qa\',\'예산은?\',[self.c])\n        self.saved={\'status\':\'not_generated\',\'execution_status\':\'validation_failed\',\'claims\':[],\n                    \'source_chunks\':[self.c],\'active_doc_ids\':[\'d1\'],\n                    \'retrieval\':{\'comparison\':False,\'standalone_query\':\'예산은?\'},\n                    \'generation\':{k:v for k,v in generated.items() if k!=\'payload\'}}\n        self.path=self.root/\'result_003.json\';self.path.write_text(json.dumps(self.saved))\n        self.cache=self.root/\'phase4_outputs/generation_cache/grounded_responses.sqlite3\'\n\n    def test_replay_works_without_api_key_and_does_not_call_sdk(self):\n        before=self.path.read_bytes()\n        with patch.dict(os.environ,{},clear=True):\n            r=replay_saved_result(self.root,self.path,registry=self.reg)\n        self.assertEqual(r[\'status\'],\'answered\');self.assertTrue(r[\'validation\'][\'valid\'])\n        self.assertEqual(r[\'replay\'][\'additional_api_calls\'],0)\n        self.assertEqual(r[\'generation\'][\'usage\'][\'api_calls\'],0)\n        self.assertEqual(r[\'generation\'][\'original_usage\'][\'api_calls\'],1)\n        self.assertEqual(self.responses.calls,1)\n        self.assertEqual(self.path.read_bytes(),before)\n        self.assertTrue(Path(r[\'revalidated_result_path\']).is_file())\n\n    def test_tampered_checksum_is_rejected_without_network(self):\n        with sqlite3.connect(self.cache) as db:db.execute("UPDATE responses SET checksum=\'bad\'")\n        with self.assertRaisesRegex(ValueError,\'무결성\'):\n            replay_saved_result(self.root,self.path,registry=self.reg,persist=False)\n\n    def test_changed_saved_context_and_query_are_rejected(self):\n        for key in [\'query\',\'text\',\'doc\']:\n            s=copy.deepcopy(self.saved)\n            if key==\'query\':s[\'retrieval\'][\'standalone_query\']=\'다른 질문\'\n            elif key==\'text\':s[\'source_chunks\'][0][\'text\']=\'다른 원문\'\n            else:s[\'source_chunks\'][0][\'doc_id\']=\'other\'\n            self.path.write_text(json.dumps(s))\n            with self.assertRaises(ValueError):replay_saved_result(self.root,self.path,registry=self.reg,persist=False)\n\n    def test_missing_cache_does_not_fallback_to_api(self):\n        self.cache.unlink()\n        with self.assertRaises(FileNotFoundError):replay_saved_result(self.root,self.path,registry=self.reg)\n        self.assertEqual(self.responses.calls,1)\n\n    def test_repeated_replay_keeps_original_result_and_cache_bytes(self):\n        before=self.cache.read_bytes()\n        a=replay_saved_result(self.root,self.path,registry=self.reg)\n        b=replay_saved_result(self.root,self.path,registry=self.reg)\n        self.assertNotEqual(a[\'revalidated_result_path\'],b[\'revalidated_result_path\'])\n        self.assertEqual(self.cache.read_bytes(),before)\n\n\nif __name__==\'__main__\':unittest.main()\n'}
namespace = {'__name__': 'citation_patch_installer', '__file__': str(ROOT / '_embedded_citation_installer.py')}
exec(compile(INSTALLER_SOURCE, '<reviewed citation installer>', 'exec'), namespace)
patch_report = namespace['apply_patch'](ROOT, embedded_files=PATCH_FILES, manifest=PATCH_MANIFEST)
print(json.dumps(patch_report, ensure_ascii=False, indent=2))


{
  "patch_id": "phase4-citation-fix-v1",
  "modified_files": [
    "rfp_phase4/citation_recovery.py",
    "rfp_phase4/rag.py",
    "rfp_phase4/summarization.py",
    "tests/test_phase4_citation_recovery.py",
    "notebooks/05_younggwan_phase4_grounded_answers.ipynb"
  ],
  "already_applied_files": [],
  "backup_directory": "/Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/patch_backups/20261111T092757_097154Z",
  "notebook_diagnostics_added": true,
  "data_or_index_modified": false,
  "cache_modified": false,
  "api_calls": 0,
  "restart_phase4_notebook_kernel_before_reusing_old_sessions": true
}


## 2. 이미 받은 모델 응답을 캐시에서 재검증

아래 셀은 기존 요청 지문·캐시 체크섬·원래 질문·입력 근거가 일치하는지 확인합니다.
검증된 답변만 표시하고, 원문 내용이 달라 실패하면 세부 인용 진단을 보여줍니다.
기존 `result_003.json` 옆에 새 `_revalidated_시각.json`을 저장합니다.
캐시가 없으면 추가 호출을 하지 않고 멈춥니다. 원래 실패 파일과 캐시 파일을 삭제하지 마세요.


In [3]:
import importlib
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
recovery = importlib.import_module('rfp_phase4.citation_recovery')
if Path(recovery.__file__).resolve().parents[1] != ROOT:
    raise RuntimeError('다른 프로젝트 모듈이 로딩되어 있습니다. 커널을 재시작한 뒤 이 노트북을 다시 실행하세요.')
recovery = importlib.reload(recovery)
replayed = recovery.replay_saved_result(ROOT, ROOT / FAILED_RESULT)
print(replayed['markdown'])
print('상태:', replayed['status'], '/', replayed['execution_status'])
print('공백 차이 복원:', replayed['validation'].get('citation_recovery', {}).get('corrected_citations', 0))
print('추가 API 호출:', replayed['replay']['additional_api_calls'])
if not replayed['validation']['valid']:
    print('실패 사유:', replayed['validation']['errors'])
    print('아래 인용 데이터는 검증에 실패한 진단용 자료입니다.')
    bad = [d for d in replayed['validation'].get('citation_diagnostics', [])
           if d.get('outcome') not in {'already_exact', 'whitespace_only'}]
    print(json.dumps(bad, ensure_ascii=False, indent=2))
print('새 결과:', replayed.get('revalidated_result_path'))


인용 근거 검증을 통과하지 못해 생성 답변을 표시하지 않았습니다.
상태: not_generated / validation_failed
공백 차이 복원: 0
추가 API 호출: 0
실패 사유: ['인용문이 청크 원문과 정확히 일치하지 않습니다.']
아래 인용 데이터는 검증에 실패한 진단용 자료입니다.
[
  {
    "claim_index": 1,
    "citation_index": 2,
    "claim_doc_id": "1G2at8Un7y8JIn5awuVxqV7OOKIj7zT4B",
    "chunk_id": "rfp:8c293ac2bb077438:a46bd452cee9dee85468de7f",
    "model_quote": "<부가세 포함, 금액단위 : 원>\n구       분\n금      액\n비     고\n용 역 비\n부가가치세 포함\n장  비\n구입비\nHW 구매 부문\n부가가치세 포함",
    "quote_is_model_data_not_a_verified_claim": true,
    "outcome": "non_whitespace_characters_differ",
    "cited_chunk_preview": "<부가세 포함, 금액단위 : 원>\n발주자\n한국주택금융공사\n직전도급자\n-\n하도급\n주는자\n사업명\n계약금액\n상  호\n대표자\n발주자와\n계약기간\n하도급되는 사업부문이 차지하는 금액(A)\n하도급\n받는자\n하수급사업명\n상  호\n대표자\n수급기간\n수급금액(B)\n하도급율\nB/A\n구       분\n금      액\n비     고\n용 역 비\n부가가치세 포함\n장  비\n구입비\nHW 구매 부문\n부가가치세 포함\nSW 구매 부문\n부가가치세 포함\n기 타 부 문\n부가가치세 포함\n합       계\n부가가치세 포함\n붙임 : 가격산출 세부내역\n※ 참고사항\n- ｢행정기관 및 "
  }
]
새 결과: /Users/codeit/Desktop/workspace_mac/중급 프로젝ᄐ

## 3. 기존 실습으로 복귀

재검증이 통과하면 기존 05 노트북 커널을 재시작하고 비교 셀까지 순서대로 실행하세요.
새 코드가 캐시의 원래 응답을 다시 검사합니다. 캐시 재사용 여부는 결과의 `generation.cache_hit`과 사용량에서 확인합니다.
여전히 실패하면 방금 저장한 `_revalidated_...json`에는 실패 인용문과 위치 진단이 포함되어 있으므로 그 파일로 원인을 더 좁힐 수 있습니다.

출처 복원이 성공해도 인용문이 주장의 의미 전체를 뒷받침하는지는 별도 검토해야 합니다.
